In [1]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

In [2]:
esconv_train = pd.read_parquet("../data/processed/esconv/train.parquet")
esconv_val = pd.read_parquet("../data/processed/esconv/val.parquet")
esconv_test = pd.read_parquet("../data/processed/esconv/test.parquet")
esconv = pd.concat([esconv_train, esconv_val, esconv_test], ignore_index=True)

helpguide = pd.read_parquet("../data/scraped/helpguide.parquet")

print(f"ESConv: {len(esconv)} rows")
print(f"Helpguide: {len(helpguide)} rows")

ESConv: 38365 rows
Helpguide: 14 rows


In [3]:
sys_turns = esconv[(esconv['speaker'] == 'sys') & (esconv['strategy'].notna())].copy()

kb_esconv = sys_turns.groupby('conv_id').apply(
    lambda x: {
        'text': ' '.join(x['text'].tolist()),
        'emotion': x['emotion_type'].iloc[0],
        'problem': x['problem_type'].iloc[0],
        'strategies': ', '.join(x['strategy'].unique().tolist()),
        'source': 'esconv'
    }, include_groups=False
).tolist()

kb_esconv_df = pd.DataFrame(kb_esconv)
print(f"ESConv knowledge base: {len(kb_esconv_df)} documents")
print(kb_esconv_df.head(2))

ESConv knowledge base: 910 documents
                                                text     emotion  \
0  Hi, good afternoon. Losing a job is always anx...     anxiety   
1  How are you doing today? I am well, what's on ...  depression   

              problem                                         strategies  \
0          job crisis  Question, Reflection of feelings, Restatement ...   
1  ongoing depression  Question, Affirmation and Reassurance, Restate...   

   source  
0  esconv  
1  esconv  


In [4]:
kb_helpguide_df = helpguide[['category', 'text', 'source']].copy()
kb_helpguide_df = kb_helpguide_df.rename(columns={'category': 'emotion'})
kb_helpguide_df['problem'] = None
kb_helpguide_df['strategies'] = None

print(f"Helpguide knowledge base: {len(kb_helpguide_df)} documents")
print(kb_helpguide_df.head(2))

Helpguide knowledge base: 14 documents
   emotion                                               text  \
0  anxiety  Do you worry excessively or feel tense and anx...   
1  anxiety  Are you plagued by constant worries, fears, an...   

                                              source problem strategies  
0  https://www.helpguide.org/mental-health/anxiet...    None       None  
1  https://www.helpguide.org/mental-health/anxiet...    None       None  


In [6]:
kb_combined = pd.concat([
    kb_esconv_df[['text', 'emotion', 'problem', 'strategies', 'source']],
    kb_helpguide_df[['text', 'emotion', 'problem', 'strategies', 'source']]
], ignore_index=True)

print(f"Total knowledge base: {len(kb_combined)} documents")
print(kb_combined['source'].value_counts())

Total knowledge base: 924 documents
source
esconv                                                                                        910
https://www.helpguide.org/mental-health/anxiety/generalized-anxiety-disorder-gad                1
https://www.helpguide.org/mental-health/anxiety/how-to-stop-worrying                            1
https://www.helpguide.org/mental-health/anxiety/panic-attacks-and-panic-disorders               1
https://www.helpguide.org/mental-health/anxiety/social-anxiety-disorder                         1
https://www.helpguide.org/mental-health/depression/coping-with-depression                       1
https://www.helpguide.org/mental-health/depression/depression-in-older-adults                   1
https://www.helpguide.org/mental-health/stress/stress-management                                1
https://www.helpguide.org/mental-health/stress/coping-with-financial-stress                     1
https://www.helpguide.org/mental-health/stress/burnout-prevention-and-recov

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding documents...")
texts = kb_combined['text'].tolist()
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)

print(f"Embeddings shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding documents...


Batches:   0%|          | 0/29 [00:00<?, ?it/s]

Embeddings shape: (924, 384)


In [8]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings.astype(np.float32))

print(f"FAISS index size: {index.ntotal} documents")

FAISS index size: 924 documents


In [9]:
def retrieve(query, top_k=5):
    query_embedding = model.encode([query]).astype(np.float32)
    distances, indices = index.search(query_embedding, top_k)
    results = kb_combined.iloc[indices[0]].copy()
    results['distance'] = distances[0]
    return results[['emotion', 'problem', 'strategies', 'source', 'distance', 'text']]

result = retrieve("I feel so anxious about my job and I can't sleep")
print(result[['emotion', 'problem', 'strategies', 'distance']])

        emotion             problem  \
219  depression  ongoing depression   
27      anxiety          job crisis   
892     anxiety          job crisis   
251     anxiety          job crisis   
310     anxiety          job crisis   

                                            strategies  distance  
219      Question, Restatement or Paraphrasing, Others  0.623329  
27   Others, Question, Affirmation and Reassurance,...  0.701237  
892  Question, Restatement or Paraphrasing, Affirma...  0.735835  
251  Question, Self-disclosure, Providing Suggestio...  0.758765  
310  Question, Information, Providing Suggestions, ...  0.773430  


In [ ]:
import os

os.makedirs("../data/rag", exist_ok=True)

faiss.write_index(index, "../data/rag/knowledge_base.index")
kb_combined.to_parquet("../data/rag/knowledge_base.parquet", index=False)

with open("../data/rag/embeddings.pkl", 'wb') as f:
    pickle.dump(embeddings, f)

print("Saved")

Saved!
  - knowledge_base.index
  - knowledge_base.parquet
  - embeddings.pkl
